In [1]:
import glob
import os
import json
import re
import pandas as pd

In [ ]:
# Define the base location and the regex pattern for splitting file paths
split_key = '[/\\\\]'
base_location = '/Users/prkarnik/Documents/Learning/DE'

In [ ]:
# Get column names from the schema for a given dataset name and sort them by the specified sorting key
def get_column_names(schemas, ds_name, sorting_key='column_position'):
    column_details = schemas[ds_name]
    columns = sorted(column_details, key=lambda col: col[sorting_key])
    return [col['column_name'] for col in colu

In [ ]:
# Read a CSV file and return a pandas DataFrame with the appropriate column names based on the schema
def read_csv(file, schemas):
    file_path_list = re.split(split_key, file)
    ds_name = file_path_list[-2]
    file_name = file_path_list[-1]
    columns = get_column_names(schemas, ds_name)
    df = pd.read_csv(file, names=columns)
    return df

In [ ]:
# Convert dataframe to json and save it to the target directory
def to_json(df, tgt_base_dir, ds_name, file_name):
    json_file_path = f'{tgt_base_dir}/{ds_name}/{file_name}.json'
    os.makedirs(f'{tgt_base_dir}/{ds_name}', exist_ok=True)
    df.to_json(
        json_file_path,
        orient='records',
        lines=True

In [ ]:
# Convert all files in the source directory to json and save them to the target directory
def file_converter(src_base_dir, tgt_base_dir, ds_name):
    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    files = glob.glob(f'{src_base_dir}/{ds_name}/part-*')

    for file in files:
        df = read_csv(file, schemas)
        file_name = re.split(split_key, file)[-1]
        to_json(df, tgt_base_dir, ds_name, file_name)

In [ ]:
# Process all files in the source directory and save them to the target directory
def process_files(ds_names=None):
    src_base_dir = f'{base_location}/data/retail_db'
    tgt_base_dir = f'{base_location}/ffc/data/retail_db_json'
    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    if not ds_names:
        ds_names = schemas.keys()
    for ds_name in ds_names:
        print(f'Processing {ds_name}')
        file_converter(src_base_dir, tgt_base_dir, ds_name)

In [ ]:
ds_name = 'orders'

file_converter(ds_name)

In [ ]:
# Run the file processing function if this script is executed directly
if __name__ == "__main__":
    process_files()